<table style="width: 100%; margin-left: auto; margin-right: auto; background-color: white;">
<tbody>
<tr>

<td style="text-align: center; width: 12%;"><img src=img/brasil-brasao.png alt="" /></td>

<td style="width: 68%;">
<h2 style="text-align: center; margin-top: 1%;">Rep&uacute;blica Federativa do Brasil</h2>
<h2 style="text-align: center; margin-top: 1%;">Minist&eacute;rio da Educa&ccedil;&atilde;o</h2>
<h2 style="text-align: center; margin-top: 1%;">Universidade Federal do Amazonas</h2>
<h2 style="text-align: center; margin-top: 1%;">Instituto de Computa&ccedil;&atilde;o</h2>
</td>

<td style="width: 12%;"><img src="img/UFAM-logo.png" alt="" /></td>

</tr>
</tbody>
</table>

---

# Ciência de Dados
## Trabalho Prático II: Modelagem de Dados

_Professor Dr. Marco Cristo_

**Alunos:**  `Sarah Lima, Nome Sobrenome e Nome Sobrenome` <br>
**E-mail:** `sarah.lima@icomp.ufam.edu.br, nome.sobrenome@icomp.ufam.edu.br e nome.sobrenome@icomp.ufam.edu.br`

In [83]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

import sklearn
import sys
import os 

In [84]:
print("Python", sys.version)
print("--------")
print("Pandas:", pd.__version__)
print("Numpy:", np.__version__)
print("SKLearn:", sklearn.__version__)

Python 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
--------
Pandas: 2.3.3
Numpy: 2.3.2
SKLearn: 1.7.2


#### __Preliminares__: Inicialmente, vamos preparar os dados conforme estabelecido no primeiro trabalho prático. Para tanto, obtenha os dados das cidades dos membros da sua equipe no diretório `data`.

In [85]:
DATA_PQT = 'data/enem2019.parquet' 
MUNICIPIOS = ['Londrina', 'Manaus', 'Rio de Janeiro'] # Use o que foi especificado para os membros da sua equipe. 

In [86]:
# Caminho do arquivo
df_enem = pd.read_parquet(DATA_PQT)
df_enem.shape

(599982, 56)

In [87]:
# Remove todas as instâncias que não possuem valores para as colunas NO_MUNICIPIO_NASCIMENTO ou SG_UF_NASCIMENTO.
# Para as colunas CO_PROVA_CN, CO_PROVA_CH, CO_PROVA_LC, CO_PROVA_MT e TP_STATUS_REDACAO, remove apenas as instâncias 
# onde todos esses valores encontram-se ausentes.
crit = ((df_enem.CO_PROVA_CN.isnull())&(df_enem.CO_PROVA_CH.isnull())&
        (df_enem.CO_PROVA_LC.isnull())&(df_enem.CO_PROVA_MT.isnull())&
        (df_enem.TP_STATUS_REDACAO.isnull()))
df_enem = df_enem[~crit]
display(df_enem.shape)

notas = [c for c in df_enem.columns if 'NU_NOTA' in c]
for c in notas:
    df_enem[c] = df_enem[c].fillna(0)
    
cods = [c for c in df_enem.columns if 'CO_PROVA' in c]
for c in cods:
    df_enem[c] = df_enem[c].fillna(500)

# Usando campo NU_IDADE, é criado FAIXA_ETARIA com os seguintes intervalos
# Instâncias com idade inferior a 10 ou superior a 100 anos sao removidas
def to_faixas(idade):
    if idade<18: return '10-18'
    elif idade<22: return '18-22'
    elif idade<26: return '22-26'
    elif idade<33: return '26-33'
    elif idade<40: return '33-40'
    elif idade<55: return '40-55'
    elif idade<63: return '55-65'
    elif idade<70: return '65-70'
    else: return '70-100'

df_enem = df_enem[(df_enem.NU_IDADE>=10)&(df_enem.NU_IDADE<=100)]
df_enem['FAIXA_ETARIA'] = df_enem.NU_IDADE.apply(to_faixas)

display(df_enem.FAIXA_ETARIA.value_counts())

nota_media = df_enem.NU_NOTA_REDACAO.mean()
df_enem['TARGET'] = df_enem.NU_NOTA_REDACAO.apply(lambda v: 1 if v>nota_media else 0)

cols_to_use = ['FAIXA_ETARIA', 'NO_MUNICIPIO_RESIDENCIA', 'TP_SEXO', 'TP_ESTADO_CIVIL', 
               'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU', 'TP_ESCOLA', 'CO_PROVA_CH',
               'CO_PROVA_LC', 'NU_NOTA_CH', 'NU_NOTA_LC', 'TP_LINGUA', 'Q001', 
               'Q002', 'Q003', 'Q004', 'Q005', 'Q006', 'Q021', 'Q022', 'Q024', 'Q025']

df_enem_c = df_enem[df_enem.NO_MUNICIPIO_PROVA.isin(MUNICIPIOS)][cols_to_use + ['TARGET']].copy()

(461570, 56)

FAIXA_ETARIA
18-22     193068
10-18     121679
22-26      59240
26-33      42011
33-40      22444
40-55      20065
55-65       2462
65-70        495
70-100        99
Name: count, dtype: int64

<div class="alert alert-info" >
<b>1 - MODELAGEM:</b> Usando o dataset <b>df_enem_c</b>, o seu objetivo é prever que alunos vão obter uma nota na redação acima da nota média observada nas cidades associadas com a sua equipe. Para tanto, prepare um pipeline de comparação justo entre os métodos, onde:
    
- Variáveis categóricas devem ser codificadas com One-hot;
- Variáveis numéricas devem ser padronizadas (z-score);
- Todos os métodos comparados devem ser otimizados (usando uma ténica de busca em _grid_ e validação cruzada) de acordo com o conjunto _mínimo_ de hiper-parâmetros descritos na tabela abaixo:

| Método | Número mínimo de parâmetros a serem avaliados |
|-----------|-------|
| __Naive Bayes__ |  |
| __Logistic Regression__ | - __C__: 0.1, 1, 10<br>- __penalty__: L2<br>- __solver__: lbfgs, liblinear |
| __Decision Tree__ | - __max_depth__: None, 8, 16<br>- __min_samples_leaf__: 1, 5, 18 |
| __Support Vector Machine__ | - __C__: 0.5, 1, 4<br>- __kernel__: linear, rbf<br>- __gamma__: scale |
| __Neural Network__ | - __hidden_layer_sizes__: (64,), (128,)<br>- __activation__: relu, tanh<br>- __alpha__: 1e-4, 1e-3<br>- __learning_rate_init__: 1e-3 |
| __Random Forest__ | - __n_estimators__: 200, 400<br>- __max_depth__: None, 12<br>- __min_samples_leaf__: 1, 5 |
| __Gradient Boosting__ | - __learning_rate__: 0.05, 0.1<br>- __max_depth__: None, 8, 16<br>- __l2_regularization__: 0.0, 1.0 |

<b>2 - AVALIAÇÃO COMPARATIVA:</b> Os resultados devem ser comparados usando as métricas __Acurácia__, __F1__ e __AUC__. Em particular, todos os métodos devem ser comparados entre si quanto à significância estatística das diferenças entre os AUCs obtidos. Para isso, use o teste não paramétrico de Wilcoxon. Ao fim, as seguintes tabelas devem ser preenchidas: 

| Método | Acurácia | F1 | AUC |
|--------|----------|----|-----|
| Naive Bayes |  |  |  |
| Logistic Regression |  |  |  |
| Decision Tree |  |  |  |
| SVM |  |  |  |
| Neural Networks |  |  |  |
| Random Forest |  |  |  |
| Gradient Boosting |  |  |  |


| Método 1 | Método 2 | p-value (diferença de AUCs) |
|-----------|-------|-------|
| Naive Bayes | Logistic Regression |  |
| Naive Bayes | Decision Tree |  |
| Naive Bayes | SVM |  |
| Naive Bayes | Neural Networks |  |
| Naive Bayes | Random Forest |  |
| Naive Bayes | Gradient Boosting |  |
| Logistic Regression  | Decision Tree |  |
| Logistic Regression | SVM |  |
| Logistic Regression | Neural Networks |  |
| Logistic Regression  | Random Forest |  |
| Logistic Regression  | Gradient Boosting |  |
| Decision Tree | SVM |  |
| Decision Tree | Neural Networks |  |
| Decision Tree  | Random Forest |  |
| Decision Tree  | Gradient Boosting |  |
| SVM | Neural Networks |  |
| SVM  | Random Forest |  |
| SVM  | Gradient Boosting |  |
| Neural Networks  | Random Forest |  |
| Neural Networks  | Gradient Boosting |  |
| Random Forest  | Gradient Boosting |  |

<b>3 - EXPLICAÇÃO:</b> Finalmente, reporte a contribuição de cada variável usada pelo __melhor modelo__, usando o SHAP (use summary plots). Ao reportar o nome das variáveis, use nomes interpretáveis (ex: EducacaoDePaiOuResponsavel em lugar de Q001) 

</div>

## - Questão 1

#### Dividindo os dados em treino e teste

In [88]:
from sklearn.model_selection import train_test_split

X = df_enem_c.drop(columns=['TARGET'])  # Variáveis independentes
y = df_enem_c['TARGET']  # Variável dependente

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Pré-processamento

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Separando os dados
X = df_enem_c.drop(columns=['TARGET'])
y = df_enem_c['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pré-processamento
preprocessor = make_column_transformer(
    (Pipeline([
        SimpleImputer(strategy='mean'),
        StandardScaler()
    ]), num_features),
    
    (Pipeline([
        SimpleImputer(strategy='most_frequent'),
        OneHotEncoder(handle_unknown='ignore')
    ]), cat_features)
)

# Pipeline completo
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Hiperparâmetros
param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['lbfgs', 'liblinear']
}

# GridSearch com validação cruzada
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc', error_score='raise')
grid_search.fit(X_train, y_train)

# Avaliação
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Acurácia:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))

Acurácia: 0.7397444519166106
F1: 0.7237687366167024
AUC: 0.8198134370511434


#### Processo de Modelagem

Regressão Logística

In [90]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Definindo o modelo e os hiperparâmetros
model = LogisticRegression()
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear']
}

# Realizando a busca em grade
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc', error_score='raise')
print(X_train[cat_features].dtypes)
print(X_train[cat_features].nunique())
print(X_train[cat_features].isnull().sum())
grid_search.fit(X_train, y_train)

# Melhor modelo
best_model = grid_search.best_estimator_

FAIXA_ETARIA       object
TP_SEXO            object
TP_ESTADO_CIVIL     int64
dtype: object
FAIXA_ETARIA       9
TP_SEXO            2
TP_ESTADO_CIVIL    5
dtype: int64
FAIXA_ETARIA       0
TP_SEXO            0
TP_ESTADO_CIVIL    0
dtype: int64


ValueError: could not convert string to float: '26-33'

#### Avaliação

Regressão Linear

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Acurácia: {accuracy}, F1: {f1}, AUC: {auc}")

NameError: name 'best_model' is not defined